# เซิร์ฟเวอร์เจนลายผ้าสด บน Colab

เปิด LoRA เป็น API แล้วต่อ tunnel ออกมาให้เว็บบนเครื่องคุณเรียกได้
ทุกครั้งที่กดเจน จะได้ภาพใหม่สดจาก GPU ไม่มีการเก็บภาพสำเร็จรูปไว้เลย

**ก่อนเริ่ม** Runtime → Change runtime type → **T4 GPU**

รันได้เลยแม้ยังไม่มี LoRA — จะใช้ SDXL เปล่าไปก่อน
จะได้ทดสอบเส้นทางทั้งหมดก่อนเทรนเสร็จ


## 1. ติดตั้ง


In [ ]:
!pip install -q 'diffusers[torch]==0.31.0' transformers==4.46.3 accelerate==1.1.1 \
    peft==0.13.2 safetensors fastapi 'uvicorn[standard]' pydantic

import torch
assert torch.cuda.is_available(), 'ยังไม่ได้เปิด GPU — Runtime > Change runtime type > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))


## 2. อัปโหลด serve.py และ LoRA

`serve.py` อยู่ในโฟลเดอร์ `training/` ของโปรเจกต์
ส่วน `lora_weights.zip` คือไฟล์ที่ได้จาก `colab_train.ipynb`

ถ้ายังไม่มี LoRA ให้อัปโหลดแค่ `serve.py` แล้วข้ามไปเซลล์ถัดไปได้เลย


In [ ]:
from google.colab import files
import pathlib, zipfile

uploaded = files.upload()

LORA_DIR = None
for name in uploaded:
    if name.endswith('.zip'):
        LORA_DIR = '/content/lora'
        pathlib.Path(LORA_DIR).mkdir(exist_ok=True)
        with zipfile.ZipFile(name) as archive:
            archive.extractall(LORA_DIR)

if LORA_DIR:
    found = list(pathlib.Path(LORA_DIR).rglob('*.safetensors'))
    print('พบไฟล์ LoRA', len(found), 'ไฟล์')
    for path in found[:5]:
        print(' ', path.relative_to(LORA_DIR))
else:
    print('ยังไม่มี LoRA จะใช้ SDXL เปล่า')


## 3. เลือก checkpoint ที่จะเสิร์ฟ

ถ้า zip มีหลาย checkpoint ให้เลือกตัวที่ `evaluate.py` บอกว่าดีที่สุด


In [ ]:
import pathlib

if LORA_DIR:
    options = sorted({p.parent for p in pathlib.Path(LORA_DIR).rglob('*.safetensors')})
    for index, path in enumerate(options):
        print(index, path)
    PICK = len(options) - 1      # เปลี่ยนเลขนี้เพื่อเลือก checkpoint อื่น
    LORA_PATH = str(options[PICK])
    print('\nจะเสิร์ฟ:', LORA_PATH)
else:
    LORA_PATH = None


## 4. เปิดเซิร์ฟเวอร์ + tunnel


In [ ]:
import subprocess, threading, time, re, os, sys

# cloudflared quick tunnel ไม่ต้องสมัครบัญชี
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

command = [sys.executable, 'serve.py', '--port', '8000']
if LORA_PATH:
    command += ['--lora', LORA_PATH]

server = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

def pump(process, label):
    for line in process.stdout:
        print(label, line.rstrip())

threading.Thread(target=pump, args=(server, '[serve]'), daemon=True).start()

# รอให้โมเดลโหลดเสร็จก่อนค่อยเปิด tunnel
import urllib.request, json
for _ in range(120):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2) as response:
            print('\nเซิร์ฟเวอร์พร้อม:', json.loads(response.read()))
            break
    except Exception:
        time.sleep(5)
else:
    raise RuntimeError('เซิร์ฟเวอร์ไม่ขึ้นภายในเวลาที่กำหนด ดู log ข้างบน')


In [ ]:
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

PUBLIC_URL = None
for line in tunnel.stdout:
    match = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if match:
        PUBLIC_URL = match.group(0)
        break

print('\n' + '=' * 60)
print('ที่อยู่สำหรับให้เว็บเรียก:')
print(PUBLIC_URL)
print('=' * 60)
print('\nเอาไปใส่ใน .env.local บนเครื่องคุณ:')
print('LORA_API_URL=' + str(PUBLIC_URL))
print('\nแล้วรีสตาร์ท next dev ใหม่หนึ่งครั้ง')
print('\nอย่าปิดแท็บนี้ ที่อยู่นี้ใช้ได้ตราบที่ Colab ยังรันอยู่')


## 5. ทดสอบว่าเจนได้จริง

ลองยิงหาตัวเองก่อน จะได้รู้ว่าปัญหาอยู่ที่เซิร์ฟเวอร์หรือที่เว็บ


In [ ]:
import urllib.request, json, base64, io
from PIL import Image
import matplotlib.pyplot as plt

payload = json.dumps({
    'prompt': 'isaanmudmee, naga motif with flame crest, deep indigo ground, cream figures',
    'count': 2, 'size': 1024,
}).encode()

request = urllib.request.Request(
    'http://127.0.0.1:8000/generate', data=payload,
    headers={'Content-Type': 'application/json'},
)
with urllib.request.urlopen(request, timeout=600) as response:
    result = json.loads(response.read())

print('ใช้เวลา', result['seconds'], 'วินาที · seeds', result['seeds'])
fig, axes = plt.subplots(1, len(result['images']), figsize=(5 * len(result['images']), 5))
if len(result['images']) == 1:
    axes = [axes]
for ax, data in zip(axes, result['images']):
    ax.imshow(Image.open(io.BytesIO(base64.b64decode(data.split(',', 1)[1]))))
    ax.axis('off')
plt.tight_layout(); plt.show()


## 6. ปล่อยให้รันค้างไว้

เซลล์นี้ทำให้ Colab ไม่ตัดการเชื่อมต่อ ปล่อยรันไว้ระหว่างใช้งานเว็บ
กดหยุดเมื่อเลิกใช้


In [ ]:
import time, urllib.request, json

while True:
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=5) as response:
            health = json.loads(response.read())
        print(time.strftime('%H:%M:%S'), 'เจนไปแล้ว', health['generated'], 'ภาพ', end='\r')
    except Exception as error:
        print('\nเซิร์ฟเวอร์ไม่ตอบ:', error)
        break
    time.sleep(60)


---

### ข้อจำกัดที่ต้องรู้

- ที่อยู่ของ tunnel **เปลี่ยนทุกครั้ง** ที่เปิด Colab ใหม่ ต้องแก้ `.env.local` ใหม่ทุกรอบ
- Colab ฟรีตัดการเชื่อมต่อเมื่อไม่ได้ใช้งานนาน ๆ และจำกัดเวลาต่อวัน
- เจนภาพละประมาณ 15-25 วินาทีบน T4
- ถ้าจะใช้จริงจังควรย้ายไปโฮสต์ที่อยู่ถาวร เช่น Replicate
